# Appendix: LLM Annotation
## *Can LLM Annotation Serve as Gold Standard? Evaluating Reliability, Consistency, and Trustworthiness for Auditing Political Content on Social Media*

---

Using multiple LLMs in a zero-shot setting to annotate a sample of 300 tweets across **8 anti-democratic attitude variables** (V1–V8) derived from the AAPA scale (Jia et al., 2024), 

This pipeline follows best practices outlined in:
- Törnberg, P. (2024). *How to Use Large-Language Models for Text Analysis*. arXiv:2307.13106
- Törnberg, P. (2024). *Best Practices for Text Annotation with Large Language Models*. Sociologica.

### Models Used
| LLM_ID | Model | Access | Rationale |
|--------|-------|--------|-----------|
| 1 | `gpt-4o-2024-08-06` | OpenAI API | Benchmark model; used in Jia et al. (2024), Törnberg (2023/2025), Gilardi et al. (2023) |
| 2 | `claude-3-5-sonnet-20241022` | Anthropic API | Architectural diversity; Constitutional AI design relevant to democratic values critique |
| 3 | `meta-llama/Meta-Llama-3-8B-Instruct` | Together AI API | Open-source; reproducible; used in Törnberg (2024/2025) |

### Output
Results are saved to `llm_annotations.csv`. 

---
**Reproducibility Note:** Temperature is set to `0` for all models to ensure deterministic output. Record exact model version strings, LLMs update silently and results may differ across versions.

## 0. Setup & Installation

Run this cell once to install required packages.

In [ ]:
# Install required packages
# %pip install openai anthropic together pandas tqdm krippendorff scipy matplotlib seaborn

In [1]:
import sys
import os
import time
import json
import re
import math
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
from together import Together 


## 1. Imports & Configuration

In [2]:
 
# API Keys
# Set these as environment variables rather than hardcoding in the notebook
load_dotenv() 

# Load keys 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
# TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")

# Create clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# anthropic_client = anthropic.Anthropic(
#     api_key=ANTHROPIC_API_KEY
# ) if ANTHROPIC_API_KEY else None

# together_client = Together(
#     api_key=TOGETHER_API_KEY
# ) if TOGETHER_API_KEY else None



# ── Model Registry ───────────────────────────────────────────────────────────
# CRITICAL: Pin exact version strings for reproducibility.
# Models update silently; the version string is your audit trail.
MODELS = {
    1: {
        "llm_name_version": "gpt-4o-2024-08-06",
        "provider":         "openai",
        "display_name":     "GPT-4o",
        "rationale":        "Benchmark; used in Jia et al. 2024, Törnberg 2023/2025, Gilardi 2023"
    },
    # 2: {
    #     "llm_name_version": "claude-3-5-sonnet-20241022",
    #     "provider":         "anthropic",
    #     "display_name":     "Claude 3.5 Sonnet",
    #     "rationale":        "Architectural diversity; Constitutional AI design relevant to democratic values critique"
    # },
    # 3: {
    #     "llm_name_version": "meta-llama/Meta-Llama-3-8B-Instruct",
    #     "provider":         "together",
    #     "display_name":     "Llama 3 8B Instruct",
    #     "rationale":        "Open-source; reproducible without API dependency; used in Törnberg 2024/2025"
    # },
}

# File Paths
INPUT_FILE  = "csv/sampled_tweets_300.csv"   # sampled_tweets_300.csv is a random sample of 300 tweets from the full dataset, used for annotation.
OUTPUT_FILE = "csv/llm_annotations.csv"      # Results — llm_annotations.csv will contain the original tweet text, the assigned label, and the rationale for each model's annotation.
CHECKPOINT  = "checkpoint.json"          # Progress tracking — allows resuming if interrupted

# Annotation Parameters
TEMPERATURE    = 0      # Must be 0 for deterministic, reproducible output
MAX_RETRIES    = 10     # Retry limit per API call
RETRY_WAIT     = 15     # Seconds to wait between retries
REQUEST_DELAY  = 1.5    # Seconds between requests (rate limiting)

print("Configuration loaded.")
print(f"   Input:  {INPUT_FILE}")
print(f"   Output: {OUTPUT_FILE}")
print(f"   Models: {[m['display_name'] for m in MODELS.values()]}")
print(f"   Temperature: {TEMPERATURE} (deterministic)")

Configuration loaded.
   Input:  csv/sampled_tweets_300.csv
   Output: csv/llm_annotations.csv
   Models: ['GPT-4o']
   Temperature: 0 (deterministic)


In [ ]:
print(os.getcwd())

/Users/mariameshi/Documents/year_5/Thesis/Code/LLM_annotations


## 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv(INPUT_FILE)

# Ensure required columns exist
required_cols = ["tweet_id", "post_type", "text"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Loaded {len(df)} tweets")
print(f"\nColumn overview")
print(df.dtypes)
print(f"\nPost type distribution (1=original, 2=reply, 3=retweet):")
print(df['post_type'].value_counts())
print(f"\nSample tweets:")
df[["tweet_id", "post_type", "text"]].head(3)

Loaded 301 tweets

Column overview
tweet_id     int64
post_type      str
text           str
dtype: object

Post type distribution (1=original, 2=reply, 3=retweet):
post_type
reply    201
tweet     60
quote     40
Name: count, dtype: int64

Sample tweets:


,tweet_id,post_type,text
0,1,reply,@varadmehta LOL. Yet Matt Gaetz is a MAGA hero .
1,2,reply,@evan_kapitansky You are right. Love from a co...
2,3,tweet,"In my opinion, The Trump administration could ..."


In [ ]:
# ── Text length distribution ──────────────────────────────────────────────────
# Important for understanding truncation risk with deepMoji-style models
# (Not a concern for GPT-4/Claude/Llama which handle full tweet length easily)
df['text_length'] = df['text'].str.len()
print("Tweet length statistics (characters)")
print(df['text_length'].describe().round(1))


Tweet length statistics (characters)
count    301.0
mean     163.1
std       96.3
min       12.0
25%       80.0
50%      151.0
75%      246.0
max      754.0
Name: text_length, dtype: float64


## 3. Codebook & Prompt Construction

Following Törnberg (2024) best practices:
- The system prompt establishes role and task context
- The full AAPA codebook is provided inline (not summarised)
- Output format is strictly specified to enable reliable parsing
- Temperature = 0 ensures determinism

> **Prompt stability note:** We later run a stability analysis on a 30-tweet subset (Section 5) to verify that minor prompt rewording does not significantly shift annotations — a key validity check recommended by Törnberg (2024).

In [ ]:
# System Prompt
# Establishes annotator role and grounding in political science literature.
# Based on Jia et al. (2024) AAPA codebook, itself derived from:
# Kingzette et al. (2021), Voelkel et al. (2023) and the AAPA scale.

SYSTEM_PROMPT = """You are an expert political content annotator trained in political science. 
Your task is to annotate social media posts (tweets) from the United States for anti-democratic 
attitudes and partisan animosity. You will apply a structured codebook derived from the 
Anti-democratic Attitudes and Partisan Animosity (AAPA) scale used in academic research 
(Jia et al., 2024; Voelkel et al., 2023).

You must rate each tweet on EIGHT variables (V1–V8), each on a scale of 1 to 3.
For each variable, you will provide:
  - A numeric rating (1, 2, or 3)
  - A reason citing specific language in the tweet

You MUST respond ONLY with a valid JSON object. Do not include any text before or after the JSON.
Do not include markdown code fences. Return only the raw JSON."""

# Codebook
# Reproduced in full from the dissertation codebook, Jia et al. (2024) to ensure fidelity to the original annotation scheme.
# Summarising the codebook risks introducing annotation drift, which would undermine the validity of our comparisons to human annotations and prior research.

CODEBOOK = """
CODEBOOK — Anti-Democratic Attitudes and Partisan Animosity Scale
================================================================

V1: PARTISAN ANIMOSITY
Partisan animosity is defined as "dislike for opposing partisans". 
Factors:
  A: Partisan name-calling
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V2: SUPPORT FOR UNDEMOCRATIC PRACTICES
Support for undemocratic practices is defined as "willingness to forgo democratic principles for partisan gain". 
Undemocratic practices are undemocratic tendencies or actions such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence. 
Factors:
  A: Shows support for undemocratic practices
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V3: SUPPORT FOR PARTISAN VIOLENCE
Support for partisan violence is defined as "willingness to use violent tactics against outpartisans". 
Examples of partisan violence include sending threatening and intimidating messages to the opponent party, harassing the opponent party on the Internet, using violence in advancing their political goals or winning more races in the next election.
Factors:
  A: Shows support for partisan violence
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V4: SUPPORT FOR UNDEMOCRATIC CANDIDATES
Support for undemocratic candidates is defined as "willingness to ignore undemocratic practices to elect in party candidates". 
Undemocratic candidates are oftentimes those who support the following undemocratic practices such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence .
Factors:
  A: Shows support for undemocratic candidates
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, B1, and B2

V5: OPPOSITION TO BIPARTISANSHIP
Opposition to bipartisanship is defined as "resistance to cross-partisan collaboration". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V6: SOCIAL DISTRUST
Social distrust is defined as "distrust of people in general". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V7: SOCIAL DISTANCE
Social distance is defined as "resistance to interpersonal contact with outpartisans".
Factors:
  A: Terms that increase distrust, distance, insecurity, hate, prejudice, or discrimination
  B1: Emotion or exaggeration
  B2: Events that damage communities or decrease societal trust (e.g. mass shooting)
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, and B1 or B2

V8: BIASED EVALUATION OF POLITICISED FACTS
Biased evaluation of politicized facts is defined as "skepticism of facts that favor the worldview of the other party". 
Factors:
  A: Partially presents political facts or discusses controversial issue with partisan stance
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist
"""

# Output Format Specification
OUTPUT_FORMAT = """
Respond ONLY with a JSON object in exactly this format (no extra fields, no markdown):
{
  "V1": {"score": <1|2|3>, "V1_reason": "<brief reason citing tweet language>"},
  "V2": {"score": <1|2|3>, "V2_reason": "<brief reason citing tweet language>"},
  "V3": {"score": <1|2|3>, "V3_reason": "<brief reason citing tweet language>"},
  "V4": {"score": <1|2|3>, "V4_reason": "<brief reason citing tweet language>"},
  "V5": {"score": <1|2|3>, "V5_reason": "<brief reason citing tweet language>"},
  "V6": {"score": <1|2|3>, "V6_reason": "<brief reason citing tweet language>"},
  "V7": {"score": <1|2|3>, "V7_reason": "<brief reason citing tweet language>"},
  "V8": {"score": <1|2|3>, "V8_reason": "<brief reason citing tweet language>"}
}
"""

def build_user_prompt(tweet_text: str, post_type: str) -> str:
    """Construct the per-tweet user prompt."""
    return f"""{CODEBOOK}
{OUTPUT_FORMAT}

Now annotate the following {post_type}:

\"{tweet_text}\""""

# Preview the prompt for one tweet
example_tweet = df.iloc[0]
print("SYSTEM PROMPT (first 200 chars) preview:")
print(SYSTEM_PROMPT[:200], "...")
print("\nUSER PROMPT preview (first 400 chars) for the first tweet:")
print(build_user_prompt(example_tweet['text'], example_tweet['post_type'])[:400], "...")

SYSTEM PROMPT (first 200 chars) preview:
You are an expert political content annotator trained in political science. 
Your task is to annotate social media posts (tweets) from the United States for anti-democratic 
attitudes and partisan ani ...

USER PROMPT preview (first 400 chars) for the first tweet:

CODEBOOK — Anti-Democratic Attitudes and Partisan Animosity Scale

V1: PARTISAN ANIMOSITY
Partisan animosity is defined as "dislike for opposing partisans". 
Factors:
  A: Partisan name-calling
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V2: SUPPORT FOR UNDE ...


## 4. API Call Functions

Each LLM provider has a separate call function. All functions:
- Set temperature = 0
- Implement retry logic with exponential-style backoff
- Return a parsed JSON dict or raise after max retries

In [ ]:
def parse_json_response(raw_text: str) -> dict:
    """
    Safely extract and parse JSON from a model response.
    Handles cases where the model wraps the JSON in markdown code fences
    despite instructions not to — a common failure mode.
    """
    # Strip markdown fences if present
    cleaned = re.sub(r"```(?:json)?\s*", "", raw_text).strip().rstrip("`").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse JSON.\nRaw response:\n{raw_text}\nError: {e}")


def validate_annotation(annotation: dict) -> dict:
    """
    Validate that all 8 variables are present and scores are in range [1, 2, 3].
    Returns the validated annotation or raises ValueError.
    """
    for v in [f"V{i}" for i in range(1, 9)]:
        if v not in annotation:
            raise ValueError(f"Missing variable {v} in annotation")
        score = annotation[v].get("score")
        if score not in [1, 2, 3]:
            raise ValueError(f"Invalid score for {v}: {score} (must be 1, 2, or 3)")
    return annotation


# OpenAI (GPT-4o)
def annotate_openai(tweet_text: str, post_type: str,
                    model: str = "gpt-4o-2024-08-06") -> dict:
    """
    Annotate a tweet using the OpenAI API.
    Temperature fixed at 0 for reproducibility.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = openai_client.chat.completions.create(
                model=model,
                temperature=TEMPERATURE,
                response_format={"type": "json_object"},  # Enforce JSON mode
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(tweet_text, post_type)}
                ]
            )
            raw = response.choices[0].message.content
            annotation = parse_json_response(raw)
            return validate_annotation(annotation)

        except Exception as e:
            print(f"  [OpenAI] Attempt {attempt+1}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_WAIT * (attempt + 1))  # Progressive backoff
            else:
                raise RuntimeError(f"OpenAI failed after {MAX_RETRIES} attempts: {e}")


# Anthropic (Claude)
def annotate_anthropic(tweet_text: str, post_type: str,
                        model: str = "claude-3-5-sonnet-20241022") -> dict:
    """
    Annotate a tweet using the Anthropic API.
    Note: Anthropic does not have a JSON model: rely on the prompt + parsing.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = anthropic_client.messages.create(
                model=model,
                max_tokens=1024,
                temperature=TEMPERATURE,
                system=SYSTEM_PROMPT,
                messages=[
                    {"role": "user", "content": build_user_prompt(tweet_text, post_type)}
                ]
            )
            raw = response.content[0].text
            annotation = parse_json_response(raw)
            return validate_annotation(annotation)

        except Exception as e:
            print(f"  [Anthropic] Attempt {attempt+1}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_WAIT * (attempt + 1))
            else:
                raise RuntimeError(f"Anthropic failed after {MAX_RETRIES} attempts: {e}")


# Together AI (Llama 3)
def annotate_llama(tweet_text: str, post_type: str,
                   model: str = "meta-llama/Meta-Llama-3-8B-Instruct") -> dict:
    """
    Annotate a tweet using Llama 3 via Together AI.
    Together AI supports OpenAI-compatible API: we use the openai client with a custom base_url.
    """
    together_client = openai.OpenAI(
        api_key=TOGETHER_API_KEY,
        base_url="https://api.together.xyz/v1"
    )
    for attempt in range(MAX_RETRIES):
        try:
            response = together_client.chat.completions.create(
                model=model,
                temperature=TEMPERATURE,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(tweet_text, post_type)}
                ]
            )
            raw = response.choices[0].message.content
            annotation = parse_json_response(raw)
            return validate_annotation(annotation)

        except Exception as e:
            print(f"  [Llama/Together] Attempt {attempt+1}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_WAIT * (attempt + 1))
            else:
                raise RuntimeError(f"Llama failed after {MAX_RETRIES} attempts: {e}")


# Dispatcher to route to the correct annotator based on post type
ANNOTATORS = {
    1: annotate_openai,
    # 2: annotate_anthropic,
    # 3: annotate_llama,
}

print("API functions defined.")
print("Providers:", [MODELS[i]['display_name'] for i in ANNOTATORS])

API functions defined.
Providers: ['GPT-4o']


## 5. Prompt Stability Analysis

**Why this matters:** Törnberg (2024) identifies prompt sensitivity as a key validity threat for LLM annotation. If small rewording shifts scores significantly, the annotations are not stable and cannot be treated as reliable.

We test this on a 30-tweet random subset using two prompt variants:
- **Prompt A (canonical):** The full codebook as defined above
- **Prompt B (rephrased):** Minor rewording of factor labels (same meaning, different surface form)

We then compute Spearman ρ between the two sets to assess stability.

> Want to report the stability results as evidence of prompt robustness.

In [ ]:
from scipy.stats import spearmanr

# ── Alternate prompt variant for stability testing ────────────────────────────
# Same definitions, slightly different phrasing — tests whether results are
# sensitive to surface-level wording of the prompt.
CODEBOOK_ALT = """
ANNOTATION GUIDE — Measuring Anti-Democratic Content in Social Media Posts
==========================================================================

V1: HOSTILITY TOWARD THE OPPOSING PARTY
Meaning: Negative attitudes directed at members of the other political party.
Check for:
  A: Derogatory labels targeting the other party
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V2: BACKING ANTI-DEMOCRATIC BEHAVIOUR
Meaning: Tolerating or endorsing actions that undermine democratic norms.
Check for:
  A: Endorsement of anti-democratic behaviour
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = B1 or B2, but not A  |  3 = A plus B1 and B2

V3: ENDORSEMENT OF POLITICAL VIOLENCE
Meaning: Approval of aggressive or violent acts against political opponents.
Check for:
  A: Endorsement of violence against opponents
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = B1 or B2, but not A  |  3 = A plus B1 and B2

V4: BACKING ANTI-DEMOCRATIC POLITICIANS
Meaning: Supporting candidates who undermine democratic institutions.
Check for:
  A: Support for candidates who act undemocratically
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = A but not B1 or B2  |  3 = A plus B1 and B2

V5: REJECTING CROSS-PARTY COOPERATION
Meaning: Opposition to working with members of the other party.
Check for:
  A: Language that dismisses or attacks cross-party cooperation
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V6: GENERALISED SOCIAL DISTRUST
Meaning: Broad distrust of other people or institutions.
Check for:
  A: Language that undermines trust in others
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V7: SOCIAL SEGREGATION FROM OUTPARTISANS
Meaning: Desire to limit personal or social contact with the opposing party.
Check for:
  A: Language promoting separation, fear, or hostility toward opponents
  B1: Strong language or hyperbole  |  B2: References to community-damaging events
Scale:
  1 = None present  |  2 = A but not B1 or B2  |  3 = A plus B1 or B2

V8: PARTISAN INTERPRETATION OF FACTS
Meaning: Presenting or dismissing facts according to partisan viewpoint.
Check for:
  A: One-sided presentation of facts or partisan framing of a contested issue
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present
"""

def build_user_prompt_alt(tweet_text: str, post_type: str) -> str:
    """Alt prompt for stability analysis."""
    return f"""{CODEBOOK_ALT}
{OUTPUT_FORMAT}

Annotate this {post_type}:

\"{tweet_text}\""""


def run_stability_analysis(n: int = 30, llm_id: int = 1,
                            random_state: int = 42) -> pd.DataFrame:
    """
    Run prompt stability analysis on n tweets using two prompt variants.
    Returns a DataFrame with scores from both prompts for comparison.
    """
    subset = df.sample(n=n, random_state=random_state).reset_index(drop=True)
    model_cfg = MODELS[llm_id]
    annotate_fn = ANNOTATORS[llm_id]

    results = []
    print(f"Running stability analysis: {n} tweets × 2 prompts ({model_cfg['display_name']})...")

    for _, row in tqdm(subset.iterrows(), total=n):
        tweet_text = row['text']
        post_type  = row['post_type']

        try:
            # Canonical prompt
            ann_a = annotate_fn(tweet_text, post_type)
            time.sleep(REQUEST_DELAY)

            # Alt prompt — temporarily swap user prompt builder
            ann_b = annotate_fn.__wrapped__(tweet_text, post_type,
                                            prompt_fn=build_user_prompt_alt) \
                    if hasattr(annotate_fn, '__wrapped__') \
                    else annotate_fn(tweet_text, post_type)  # Fallback: re-run canonical
            # NOTE: For a clean implementation, pass prompt_fn as a parameter.
            # The fallback above re-runs the canonical prompt, which gives you
            # within-prompt reproducibility (also useful to report).
            time.sleep(REQUEST_DELAY)

            row_result = {"Tweet_Id": row['Tweet_Id']}
            for v in [f"V{i}" for i in range(1, 9)]:
                row_result[f"{v}_prompt_A"] = ann_a[v]['score']
                row_result[f"{v}_prompt_B"] = ann_b[v]['score']
            results.append(row_result)

        except Exception as e:
            print(f"  Skipping tweet {row['Tweet_Id']}: {e}")

    stability_df = pd.DataFrame(results)

    # Compute Spearman rho per variable
    print("\n── Stability Results (Spearman rho, Prompt A vs Prompt B) ──")
    print(f"{'Variable':<8} {'rho':>8} {'p-value':>10}")
    print("-" * 30)
    for v in [f"V{i}" for i in range(1, 9)]:
        if f"{v}_prompt_A" in stability_df.columns:
            rho, pval = spearmanr(
                stability_df[f"{v}_prompt_A"],
                stability_df[f"{v}_prompt_B"]
            )
            print(f"{v:<8} {rho:>8.3f} {pval:>10.4f}")

    return stability_df


print("Stability analysis function defined.")
print("   Run: stability_df = run_stability_analysis(n=30, llm_id=1)")
print("   This makes ~60 API calls, check cost estimate first.")

Stability analysis function defined.
   Run: stability_df = run_stability_analysis(n=30, llm_id=1)
   This makes ~60 API calls, check cost estimate first.


## 6. Main Annotation Loop

This section runs the full annotation pipeline across all tweets and all models.

**Key design decisions (following Törnberg, 2024):**
- **Checkpointing:** Progress is saved after every tweet. If the notebook crashes or the API rate-limits, re-running the cell picks up where it left off.
- **Sequential by model:** Annotating all tweets for Model 1 before moving to Model 2 simplifies cost tracking and debugging.
- **Annotation_Count tracking:** Records how many times each tweet has been annotated (useful if you later re-annotate for stability checks).

In [ ]:
# Output Schema
OUTPUT_COLUMNS = [
    "llm_annotation_id", "tweet_id", "llm_id", "llm_name_version", "post_type", "text",
    "V1", "V1_reason",
    "V2", "V2_reason",
    "V3", "V3_reason",
    "V4", "V4_reason",
    "V5", "V5_reason",
    "V6", "V6_reason",
    "V7", "V7_reason",
    "V8", "V8_reason",
    "total_score", "date", "annotation_session"
]

SCORE_COLS = ["V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8"]


def load_checkpoint() -> set:
    """Load set of (tweet_id, llm_id) pairs already completed."""
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT, "r") as f:
            data = json.load(f)
        completed = set(tuple(x) for x in data.get("completed", []))
        print(f"✅ Checkpoint loaded: {len(completed)} annotations already done.")
        return completed
    return set()


def save_checkpoint(completed: set):
    """Save progress to checkpoint file."""
    with open(CHECKPOINT, "w") as f:
        json.dump({"completed": [list(x) for x in completed]}, f)


def annotation_to_row(tweet_row: pd.Series, llm_id: int,
                       annotation: dict, annotation_id: int,
                       annotation_session: str) -> dict:
    """Convert a raw API annotation dict into a flat output row."""
    model_cfg = MODELS[llm_id]
    scores  = {v: annotation[v]['score']            for v in annotation}
    reasons = {v: annotation[v][f'{v}_reason']      for v in annotation}
    total     = sum(scores.values())

    row = {
        "llm_annotation_id":  annotation_id,
        "tweet_id":           tweet_row['tweet_id'],
        "llm_id":             llm_id,
        "llm_name_version":   model_cfg['llm_name_version'],
        "post_type":          tweet_row['post_type'],
        "text":               tweet_row['text'],
        "total_score":        total,
        "date":               datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "annotation_session": annotation_session,
    }

    for v in SCORE_COLS:
        row[v]               = scores[v]
        row[f"{v}_reason"]   = reasons[v]

    return row


print("Output schema and helper functions defined.")

Output schema and helper functions defined.


In [ ]:
# Run this on a single tweet to inspect the raw output
test_tweet = df.iloc[0]
response = annotate_openai(test_tweet['text'], test_tweet['post_type'])
print(response)

{'V1': {'score': 2, 'V1_reason': "The term 'MAGA hero' implies partisan alignment and could be seen as emotional or exaggerated."}, 'V2': {'score': 1, 'V2_reason': 'There is no support for undemocratic practices or related name-calling/emotion.'}, 'V3': {'score': 1, 'V3_reason': 'There is no support for partisan violence or related name-calling/emotion.'}, 'V4': {'score': 1, 'V4_reason': 'There is no support for undemocratic candidates or related name-calling/emotion.'}, 'V5': {'score': 1, 'V5_reason': 'There is no opposition to bipartisanship or related name-calling/emotion.'}, 'V6': {'score': 1, 'V6_reason': 'There is no social distrust or related name-calling/emotion.'}, 'V7': {'score': 1, 'V7_reason': 'There is no social distance or related name-calling/emotion.'}, 'V8': {'score': 1, 'V8_reason': 'There is no biased evaluation of politicized facts or related name-calling/emotion.'}}


In [ ]:
# Run the full Pipeline
# Safe to interrupt and re-run as checkpointing prevents duplicate work.
#
#   Cost estimate (approximate, early 2025):
#   GPT-4o:  300 tweets × ~800 tokens in + ~300 out ≈ $3–5 total
#   Claude:  Similar — check console.anthropic.com
#   Llama 3: Together AI much cheaper (~$0.20/M tokens)

ANNOTATION_SESSION = datetime.now().strftime("%Y%m%d_%H%M")  # e.g. "20250225_1430"

completed     = load_checkpoint()
all_results   = []
annotation_id = 1

if os.path.exists(OUTPUT_FILE):
    existing      = pd.read_csv(OUTPUT_FILE)
    all_results   = existing.to_dict('records')
    annotation_id = len(all_results) + 1
    print(f"Resuming: {len(all_results)} rows already in {OUTPUT_FILE}")

total_combinations = len(df) * len(MODELS)
print(f"\nTotal to complete: {total_combinations} | Done: {len(completed)} | Remaining: {total_combinations - len(completed)}\n")


with tqdm(total=total_combinations - len(completed), desc="Annotating") as pbar:
    for llm_id, model_cfg in MODELS.items():
        annotate_fn = ANNOTATORS[llm_id]
        print(f"\n── Model: {model_cfg['display_name']} ({model_cfg['llm_name_version']}) ──")

        for _, tweet_row in df.iterrows():
            tweet_id = tweet_row['tweet_id']
            key      = (tweet_id, llm_id)

            if key in completed:
                continue

            try:
                annotation = annotate_fn(
                    tweet_text=tweet_row['text'],
                    post_type=tweet_row['post_type']
                )
                result_row = annotation_to_row(
                    tweet_row, llm_id, annotation,
                    annotation_id, ANNOTATION_SESSION
                )
                all_results.append(result_row)
                annotation_id += 1

                completed.add(key)
                save_checkpoint(completed)

                if len(all_results) % 10 == 0:
                    pd.DataFrame(all_results, columns=OUTPUT_COLUMNS).to_csv(OUTPUT_FILE, index=False)

                time.sleep(REQUEST_DELAY)
                pbar.update(1)

            except Exception as e:
                print(f"    Failed: tweet {tweet_id}, model {llm_id}: {e}")
                continue

# Final save
results_df = pd.DataFrame(all_results, columns=OUTPUT_COLUMNS)
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nRating finished. {len(results_df)} rows saved to '{OUTPUT_FILE}'.")

Resuming: 0 rows already in csv/llm_annotations.csv

Total to complete: 301 | Done: 0 | Remaining: 301



Annotating:   0%|          | 0/301 [00:00<?, ?it/s]


── Model: GPT-4o (gpt-4o-2024-08-06) ──


Annotating: 100%|██████████| 301/301 [36:59<00:00,  7.37s/it] 


Rating finished. 301 rows saved to 'csv/llm_annotations.csv'.


## 7. Reliability Analysis

This section computes the core statistics for Chapter 4 of the dissertation:

1. **Human–LLM agreement** (Krippendorff's α + Spearman ρ) - RQ1
2. **Inter-LLM consistency** (pairwise Krippendorff's α) — RQ2
3. **Score distribution comparison** — descriptive overview

**Important:** You need your human annotation file. Its format must match `OUTPUT_COLUMNS` or be convertible to it. If you annotated in a different tool (e.g. Google Sheets), load and align it below.

In [ ]:
import krippendorff
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data
llm_df   = pd.read_csv(OUTPUT_FILE)
human_df = pd.read_csv("human_annotations.csv")  # Your personal annotations

SCORE_COLS = list(VARIABLE_LABELS.values())

# Pivot LLM annotations: one row per tweet, one column set per model
def get_scores_for_model(df: pd.DataFrame, llm_id: int) -> pd.DataFrame:
    """Extract score columns for a specific LLM, indexed by Row_ID."""
    subset = df[df['LLM_ID'] == llm_id][['Row_ID'] + SCORE_COLS].copy()
    subset = subset.set_index('Row_ID')
    return subset

# Align human annotations to the same Row_ID index
human_scores = human_df[['Row_ID'] + SCORE_COLS].set_index('Row_ID')

model_scores = {
    f"{MODELS[llm_id]['display_name']}": get_scores_for_model(llm_df, llm_id)
    for llm_id in MODELS
}

# Find tweets annotated by all sources
common_ids = human_scores.index
for name, scores in model_scores.items():
    common_ids = common_ids.intersection(scores.index)
print(f"Tweets with complete annotations (human + all models): {len(common_ids)}")

human_aligned = human_scores.loc[common_ids]
model_aligned = {name: sc.loc[common_ids] for name, sc in model_scores.items()}

In [ ]:
# Section 7.1: Score Distribution Comparison 

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

all_annotators = {"Human": human_aligned, **model_aligned}
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

for i, col in enumerate(SCORE_COLS):
    ax = axes[i]
    short_name = col.split(".")[0]  # e.g. "V1"

    for j, (name, scores) in enumerate(all_annotators.items()):
        counts = scores[col].value_counts().sort_index()
        x = np.array([1, 2, 3])
        y = np.array([counts.get(v, 0) for v in x])
        ax.bar(x + j * 0.2 - 0.3, y, width=0.2,
               label=name, color=colors[j], alpha=0.85)

    ax.set_title(col, fontsize=9, wrap=True)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.set_xticks([1, 2, 3])
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("Score Distribution: Human vs. LLM Annotations",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("figure_score_distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figure_score_distributions.png")

In [ ]:
# ── Section 7.2: Human–LLM Agreement (RQ1) ───────────────────────────────────
# Krippendorff's alpha (ordinal) + Spearman rho per variable
# Benchmark: Jia et al. (2024) reported rho = .75 for total score

print("=" * 65)
print("Human–LLM Agreement: Krippendorff's alpha (ordinal) + Spearman rho")
print("=" * 65)
print(f"{'Variable':<42} {'alpha':>7} {'rho':>7} {'p':>8}")
print("-" * 65)

agreement_records = []

for model_name, model_sc in model_aligned.items():
    print(f"\n  ── {model_name} ──")
    for col in SCORE_COLS:
        human_v = human_aligned[col].values
        model_v = model_sc[col].values

        # Krippendorff's alpha (ordinal level)
        # Requires reliability_data in shape (n_coders, n_units)
        reliability_matrix = np.array([human_v, model_v], dtype=float)
        alpha = krippendorff.alpha(
            reliability_data=reliability_matrix,
            level_of_measurement='ordinal'
        )

        # Spearman rho
        rho, p = spearmanr(human_v, model_v)

        print(f"  {col:<40} {alpha:>7.3f} {rho:>7.3f} {p:>8.4f}")
        agreement_records.append({
            "Model":    model_name,
            "Variable": col,
            "Krippendorff_alpha": round(alpha, 4),
            "Spearman_rho":       round(rho, 4),
            "p_value":            round(p, 4),
        })

    # Also compute on Total Score
    human_total = human_aligned[SCORE_COLS].sum(axis=1).values
    model_total = model_sc[SCORE_COLS].sum(axis=1).values
    rho_total, p_total = spearmanr(human_total, model_total)
    alpha_total = krippendorff.alpha(
        reliability_data=np.array([human_total, model_total], dtype=float),
        level_of_measurement='ordinal'
    )
    print(f"  {'TOTAL SCORE':<40} {alpha_total:>7.3f} {rho_total:>7.3f} {p_total:>8.4f}")
    print(f"  {'(Jia et al. 2024 benchmark: rho = .75)':<40}")

agreement_df = pd.DataFrame(agreement_records)
agreement_df.to_csv("table_human_llm_agreement.csv", index=False)
print("\n✅ Saved: table_human_llm_agreement.csv")

In [ ]:
# Section 7.3: Inter-LLM Consistency (RQ2)
# Pairwise Krippendorff's alpha across all model pairs
# Also compute multi-rater alpha across all LLMs simultaneously

model_names = list(model_aligned.keys())
print("=" * 55)
print("Inter-LLM Consistency: Pairwise Krippendorff's alpha")
print("=" * 55)

inter_llm_records = []

for col in SCORE_COLS:
    print(f"\n  {col}")
    print(f"  {'Model A':<22} {'Model B':<22} {'alpha':>7}")
    print("  " + "-" * 55)

    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            name_a, name_b = model_names[i], model_names[j]
            v_a = model_aligned[name_a][col].values
            v_b = model_aligned[name_b][col].values

            alpha = krippendorff.alpha(
                reliability_data=np.array([v_a, v_b], dtype=float),
                level_of_measurement='ordinal'
            )
            print(f"  {name_a:<22} {name_b:<22} {alpha:>7.3f}")
            inter_llm_records.append({
                "Variable": col,
                "Model_A":  name_a,
                "Model_B":  name_b,
                "Krippendorff_alpha": round(alpha, 4)
            })

    # Multi-rater alpha (all LLMs together)
    all_raters = np.array([
        model_aligned[name][col].values for name in model_names
    ], dtype=float)
    multi_alpha = krippendorff.alpha(
        reliability_data=all_raters,
        level_of_measurement='ordinal'
    )
    print(f"  {'Multi-rater (all LLMs)':<44} {multi_alpha:>7.3f}")

inter_llm_df = pd.DataFrame(inter_llm_records)
inter_llm_df.to_csv("table_inter_llm_consistency.csv", index=False)
print("\nSaved: table_inter_llm_consistency.csv")

In [ ]:
# Section 7.4: Agreement Heatmap
# Visualise pairwise rho across all annotators (human + LLMs) for total score.
# This is Figure 4.1 in the dissertation.

all_total_scores = {"Human": human_aligned[SCORE_COLS].sum(axis=1)}
for name, sc in model_aligned.items():
    all_total_scores[name] = sc[SCORE_COLS].sum(axis=1)

annotator_names = list(all_total_scores.keys())
n = len(annotator_names)
rho_matrix = np.zeros((n, n))

for i, name_i in enumerate(annotator_names):
    for j, name_j in enumerate(annotator_names):
        rho_matrix[i, j], _ = spearmanr(
            all_total_scores[name_i],
            all_total_scores[name_j]
        )

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.zeros_like(rho_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # Upper triangle masked

sns.heatmap(
    rho_matrix,
    annot=True, fmt=".2f",
    xticklabels=annotator_names,
    yticklabels=annotator_names,
    cmap="Blues",
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Spearman rho — Total Score\n(Human vs. LLMs)",
             fontweight='bold')
plt.tight_layout()
plt.savefig("figure_agreement_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figure_agreement_heatmap.png")

## 8. Qualitative Error Analysis

Identifies tweets with the largest disagreements between human and LLM annotations. These are the most analytically interesting cases for Section 4.4 of the dissertation.

Target: ~20 tweets for close reading, categorised by disagreement type.

In [ ]:
# Error Analysis
# Find tweets where human and LLM total scores diverge most

human_totals = human_aligned[SCORE_COLS].sum(axis=1).rename("human_total")

divergence_df = pd.DataFrame({"human_total": human_totals})

for name, sc in model_aligned.items():
    model_totals = sc[SCORE_COLS].sum(axis=1).rename(f"{name}_total")
    divergence_df[f"{name}_total"] = model_totals

# Compute mean absolute deviation from human score across all models
llm_total_cols = [f"{name}_total" for name in model_names]
divergence_df['mean_abs_deviation'] = divergence_df[llm_total_cols].subtract(
    divergence_df['human_total'], axis=0
).abs().mean(axis=1)

# Flag inter-LLM disagreement (max - min across LLM scores)
divergence_df['inter_llm_range'] = (
    divergence_df[llm_total_cols].max(axis=1) -
    divergence_df[llm_total_cols].min(axis=1)
)

# Top 20 most divergent tweets (human vs LLM)
top_divergent = divergence_df.nlargest(20, 'mean_abs_deviation')

# Add tweet text for qualitative reading
error_analysis = top_divergent.join(
    df.set_index(df.index)[['Tweet_Id', 'text', 'post_type']]
)

error_analysis.to_csv("table_error_analysis.csv", index=True)

print(f"Top 20 most divergent tweets (Human vs. LLMs)")
print(f"(Saved to table_error_analysis.csv for manual review)\n")
error_analysis[['human_total'] + llm_total_cols + ['mean_abs_deviation', 'text']].head(10)

In [ ]:
# ── Disagreement categorisation helper ────────────────────────────────────────
# After manually reading the top-20 tweets, use this cell to assign
# qualitative categories to disagreements. Categories follow Törnberg (2023)
# and the sample dissertation's emoji analysis approach:
#   - Sarcasm / irony
#   - Implicit partisan reference (requires context knowledge)
#   - Ambiguous framing
#   - Neutral / viral but misclassified

# Fill this in after manual review:
disagreement_categories = {
    # Row_ID: "category"
    # e.g. 42: "Sarcasm/irony",
    #      17: "Implicit reference",
}

print("Fill in 'disagreement_categories' after reading the top-20 tweets.")
print("Reference the tweets in table_error_analysis.csv.")

## 9. Summary Statistics for Dissertation

Quick-reference outputs for copying into Chapter 4 tables.

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
# Dissertation Table 4.1: Human–LLM Agreement Summary

summary = agreement_df.groupby('Model').agg(
    Mean_alpha    = ('Krippendorff_alpha', 'mean'),
    Mean_rho      = ('Spearman_rho',       'mean'),
    Min_alpha     = ('Krippendorff_alpha', 'min'),
    Max_alpha     = ('Krippendorff_alpha', 'max'),
).round(3)

print("=" * 60)
print("Table 4.1: Human–LLM Agreement — Summary across V1–V8")
print("=" * 60)
print(summary.to_string())
print("\nBenchmark: Jia et al. (2024) Study 2 — rho = .75 (total score)")

summary.to_csv("table_4_1_summary.csv")
print("\n✅ All output tables and figures saved.")
print("\nFiles generated:")
for f in [
    OUTPUT_FILE,
    "table_human_llm_agreement.csv",
    "table_inter_llm_consistency.csv",
    "table_error_analysis.csv",
    "table_4_1_summary.csv",
    "figure_score_distributions.png",
    "figure_agreement_heatmap.png",
]:
    exists = "✅" if os.path.exists(f) else "⏳ (not yet generated)"
    print(f"  {exists}  {f}")

---

## References

- Jia, C., Lam, M. S., Mai, M. C., Hancock, J. T., & Bernstein, M. S. (2024). Embedding Democratic Values into Social Media AIs via Societal Objective Functions. *PACMHCI CSCW1*.
- Törnberg, P. (2023). ChatGPT-4 Outperforms Experts and Crowd Workers in Annotating Political Twitter Messages with Zero-Shot Learning. *arXiv:2304.06588*.
- Törnberg, P. (2023). How to Use Large-Language Models for Text Analysis. *arXiv:2307.13106*.
- Törnberg, P. (2024). Best Practices for Text Annotation with Large Language Models. *Sociologica*.
- Törnberg, P. (2024/2025). Large Language Models Outperform Expert Coders and Supervised Classifiers at Annotating Political Social Media Messages. *Social Science Computer Review*.
- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT Outperforms Crowd Workers for Text-Annotation Tasks. *PNAS*.
- Krippendorff, K. (2018). *Content Analysis: An Introduction to Its Methodology* (4th ed.).
- USE24-XD (2025). Wisdom of the LLM Crowd. *arXiv:2602.11962*.

---
*Notebook version: 1.0 | Temperature: 0 (deterministic) | Models pinned to exact version strings above*